# Model Evaluation Notebook

Compare different models across 4 tasks before committing to the full pipeline.
All models accessed via OpenRouter.

1. **Transcription** — Gemini 2.5 Flash vs GPT Audio Mini vs Voxtral Small
2. **Audio Prosody** — Gemini 2.5 Flash vs GPT Audio Mini vs Voxtral Small
3. **OCR** — Gemini Flash vs Claude Sonnet
4. **Scene Description** — Gemini Flash (video clip input)

Estimated cost: ~$1-2 for all tests.

## Setup

In [23]:
import os
import sys
import base64
import time
from pathlib import Path
from dotenv import load_dotenv

_REPO_ROOT = Path("..").resolve()
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))
from google import genai
from google.genai import types
from tabulate import tabulate

load_dotenv("../.env")

# v1alpha required for media_resolution parameter
gemini_client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY"),
    http_options={"api_version": "v1alpha"},
)

SAMPLES_DIR = Path("../data/samples")
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

# Models for audio tasks (transcription + prosody)
AUDIO_MODELS = {
    "Gemini 2.5 Flash": "gemini-2.5-flash",
    "Gemini 3 Flash": "gemini-3-flash-preview",
    #"GPT Audio Mini": "openai/gpt-audio-mini",
    #"Voxtral Small": "mistralai/voxtral-small-24b-2507",
}

print("Client initialized.")

Client initialized.


## Test Samples

Provide your own audio clips and keyframes directly.

Place files in `../data/samples/` or reference them by path below.

In [24]:
# Transcription test clips
transcription_clips = [
    "../data/samples/transcript1.wav",
    "../data/samples/transcript2.wav",
    "../data/samples/transcript3.wav",
]
transcription_labels = [
    "mumbling and background noise",
    "mumbling",
    "static interruptions",
]

# Prosody test clips
prosody_clips = [
    "../data/samples/prosody1.wav",
    "../data/samples/prosody2.wav",
    "../data/samples/prosody3.wav",
]
prosody_labels = [
    "shouting",
    "crying/distressed",
    "shouting commands",
]

# OCR test frames
ocr_frames = [
    "../data/samples/ocr1.png",
    "../data/samples/ocr2.png",
    "../data/samples/ocr3.png",
]
ocr_labels = [
    "simple license plate",
    "warped license plate",
    "far and blurry license plate",
]

# Scene description test clips (video input to Gemini Flash)
scene_clips = [
    "../data/samples/scene1.mp4",
    "../data/samples/scene2.mp4",
    "../data/samples/scene3.mp4",
]
scene_labels = [
    "lady in purple, man in blue, in the day",
    "smoke, topless man in red shorts, fire truck, at night",
    "many things on bed in hotel room",
]

# Verify all files exist
all_files = transcription_clips + prosody_clips + ocr_frames + scene_clips
for f in all_files:
    assert Path(f).exists(), f"File not found: {f}"

print(f"Ready: {len(transcription_clips)} transcription clips, {len(prosody_clips)} prosody clips, "
      f"{len(ocr_frames)} OCR frames, {len(scene_clips)} scene clips")

Ready: 3 transcription clips, 3 prosody clips, 3 OCR frames, 3 scene clips


## Helper Functions

In [25]:
import subprocess
from src.providers.prompts import COMBINED_PROMPT as SCENE_PROMPT


def encode_file_b64(file_path: str) -> str:
    """Encode any file as base64."""
    with open(file_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def get_audio_format(audio_path: str) -> str:
    """Get the format string for audio input."""
    ext = Path(audio_path).suffix.lower()
    format_map = {".wav": "wav", ".mp3": "mp3", ".mp4": "mp4", ".m4a": "m4a"}
    return format_map.get(ext, ext.lstrip("."))


def get_image_media_type(image_path: str) -> str:
    """Get the MIME type for an image file."""
    ext = Path(image_path).suffix.lower()
    type_map = {".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg"}
    return type_map.get(ext, "image/png")


def get_video_media_type(video_path: str) -> str:
    """Get the MIME type for a video file."""
    ext = Path(video_path).suffix.lower()
    type_map = {".mp4": "video/mp4", ".webm": "video/webm", ".mov": "video/mov"}
    return type_map.get(ext, "video/mp4")


def compress_video_720p(video_path: str) -> str:
    """Compress video to 720p if needed. Returns path to compressed file."""
    path = Path(video_path)

    # Check current resolution
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0",
         "-show_entries", "stream=height", "-of", "csv=p=0", str(path)],
        capture_output=True, text=True,
    )
    height = int(probe.stdout.strip())

    if height <= 720:
        print(f"  {path.name}: already {height}p, skipping compression")
        return video_path

    compressed_path = path.parent / f"{path.stem}_720p{path.suffix}"
    if compressed_path.exists():
        print(f"  {path.name}: 720p version already exists")
        return str(compressed_path)

    print(f"  {path.name}: compressing {height}p -> 720p...")
    subprocess.run(
        ["ffmpeg", "-i", str(path), "-vf", "scale=-2:720",
         "-c:a", "copy", str(compressed_path), "-y", "-loglevel", "error"],
        check=True,
    )

    orig_mb = path.stat().st_size / (1024 * 1024)
    comp_mb = compressed_path.stat().st_size / (1024 * 1024)
    print(f"  {orig_mb:.1f}MB -> {comp_mb:.1f}MB")
    return str(compressed_path)


def call_gemini(model: str, contents, max_tokens: int = 2048) -> str:
    """Call a model via the Gemini API and return the text response."""
    start = time.time()
    response = gemini_client.models.generate_content(
        model=model,
        contents=contents,
        config=types.GenerateContentConfig(max_output_tokens=max_tokens),
    )
    elapsed = time.time() - start
    usage = response.usage_metadata
    print(f"  [{model}] {elapsed:.1f}s | {usage.prompt_token_count} in / {usage.candidates_token_count} out")
    return response.text


OCR_PROMPT = (
    "Extract ALL visible text from this image. Include:\n"
    "- License plates (format: PLATE: XXX-XXXX)\n"
    "- Street signs, store signs, banners\n"
    "- Badge numbers, name tags\n"
    "- Screen text, timestamps, watermarks\n"
    "- Any other readable text\n\n"
    "For each piece of text, note its location and legibility "
    "(clear / partial / blurry). If no text is visible, say 'No text detected.'"
)


def prepare_image_payload(image_path: str) -> tuple[str, str]:
    """Return (img_b64, media_type) for image messages."""
    return encode_file_b64(image_path), get_image_media_type(image_path)


def prepare_audio_payload(audio_path: str) -> tuple[str, str]:
    """Return (audio_b64, format) for audio messages."""
    return encode_file_b64(audio_path), get_audio_format(audio_path)


def ocr_with_model(model_id: str, img_b64: str, media_type: str) -> str:
    """Run OCR using pre-encoded image."""
    image_bytes = base64.b64decode(img_b64)
    content = types.Content(
        parts=[
            types.Part(inline_data=types.Blob(data=image_bytes, mime_type=media_type)),
            types.Part(text=OCR_PROMPT),
        ]
    )
    return call_gemini(model_id, content)


def prepare_scene_video_payload(video_path: str) -> tuple[str, str]:
    """Compress to 720p if needed; return (video_b64, media_type) for video messages."""
    compressed_path = compress_video_720p(video_path)
    return encode_file_b64(compressed_path), get_video_media_type(compressed_path)


def describe_scene_video_mediumres(model_id: str, video_b64: str, media_type: str) -> str:
    """Send pre-encoded video for scene description at medium resolution (70 tokens/frame)."""
    video_bytes = base64.b64decode(video_b64)
    content = types.Content(
        parts=[
            types.Part(
                inline_data=types.Blob(data=video_bytes, mime_type=media_type),
                media_resolution=types.PartMediaResolution(
                    level=types.PartMediaResolutionLevel.MEDIA_RESOLUTION_MEDIUM,
                ),
            ),
            types.Part(text=SCENE_PROMPT),
        ]
    )
    return call_gemini(model_id, content)


def describe_scene_video_highres(model_id: str, video_b64: str, media_type: str) -> str:
    """Send pre-encoded video for scene description at high resolution (280 tokens/frame)."""
    video_bytes = base64.b64decode(video_b64)
    content = types.Content(
        parts=[
            types.Part(
                inline_data=types.Blob(data=video_bytes, mime_type=media_type),
                media_resolution=types.PartMediaResolution(
                    level=types.PartMediaResolutionLevel.MEDIA_RESOLUTION_HIGH,
                ),
            ),
            types.Part(text=SCENE_PROMPT),
        ]
    )
    return call_gemini(model_id, content)

In [26]:
# Run once after Helper Functions: encode each media file once; reuse across model comparisons
transcription_audio_payloads = [prepare_audio_payload(p) for p in transcription_clips]
prosody_audio_payloads = [prepare_audio_payload(p) for p in prosody_clips]
ocr_image_payloads = [prepare_image_payload(p) for p in ocr_frames]
scene_video_payloads = [prepare_scene_video_payload(p) for p in scene_clips]
print(
    f"Prepared {len(transcription_audio_payloads)} transcription, "
    f"{len(prosody_audio_payloads)} prosody audio, "
    f"{len(ocr_image_payloads)} OCR image, "
    f"{len(scene_video_payloads)} scene video payload(s)."
)

  scene1.mp4: 720p version already exists
  scene2.mp4: 720p version already exists
  scene3.mp4: 720p version already exists
Prepared 3 transcription, 3 prosody audio, 3 OCR image, 3 scene video payload(s).


---
## Test 1: Transcription

Compare 3 models (all via OpenRouter):
- **Gemini 2.5 Flash** — 3.1% WER on benchmarks, native audio input
- **GPT Audio Mini** — cost-efficient audio model from OpenAI
- **Voxtral Small 24B** — Mistral's audio-capable model

Key question: Which handles noisy body-cam audio best?

Run the **Pre-encode media** cell (after Helper Functions) before this test so `transcription_audio_payloads` exists.

In [ ]:
TRANSCRIPTION_PROMPT = (
    "Transcribe this audio clip verbatim. Include timestamps "
    "for each sentence or phrase. Format as:\n"
    "[MM:SS] text\n\n"
    "If you can distinguish multiple speakers, label them Speaker 1, Speaker 2, etc. "
    "Include non-speech sounds in brackets like [wind noise], [radio chatter]."
)


def transcribe_with_model(model_id: str, audio_b64: str, audio_fmt: str) -> str:
    """Transcribe pre-encoded audio via Gemini API."""
    audio_bytes = base64.b64decode(audio_b64)
    mime_type = f"audio/{audio_fmt}"
    content = types.Content(
        parts=[
            types.Part(inline_data=types.Blob(data=audio_bytes, mime_type=mime_type)),
            types.Part(text=TRANSCRIPTION_PROMPT),
        ]
    )
    return call_gemini(model_id, content)

In [13]:
# Run transcription comparison across all models
transcription_results = []

for i, clip_path in enumerate(transcription_clips):
    print(f"\n{'='*60}")
    print(f"Transcription clip {i}: {transcription_labels[i]}")
    print(f"{'='*60}")

    audio_b64, audio_fmt = transcription_audio_payloads[i]
    results = {}
    for model_name, model_id in AUDIO_MODELS.items():
        print(f"\n--- {model_name} ---")
        results[model_name] = transcribe_with_model(model_id, audio_b64, audio_fmt)
        print(results[model_name])

    transcription_results.append(results)


Transcription clip 0: mumbling and background noise

--- Gemini 2.5 Flash ---


APIStatusError: Error code: 402 - {'error': {'message': 'This request requires at least $0.50 in balance for audio', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_3BK38cS122wdjLOhZmZ8z8ySdVK'}

### Transcription Notes

Write your observations here after running:
- Which model handled noisy audio better?
- Did either hallucinate words?
- Timestamp accuracy?
- Speaker separation quality?

---
## Test 2: Audio Prosody (Shouting / Raised Voice Detection)

Compare the same 3 models on detecting:
- Raised voices / shouting
- Emotional tone (calm, agitated, distressed)
- Background sounds (sirens, engines, radio)

Key concern: LLMs may exhibit "lexical dominance" — predicting emotion from
words rather than acoustic cues. Does the model detect shouting from tone,
or just from what's being said?

Requires the **Pre-encode media** cell (`prosody_audio_payloads`).

In [ ]:
PROSODY_PROMPT = """Analyze the human speech in this body-worn camera footage by assessing these five audio dimensions:
1. Average volume — overall loudness of the speech
2. Volume variation — how much the loudness changes (steady vs. dynamic)
3. Average pitch — general tone of the voice (high vs. low)
4. Pitch variation — how much the pitch modulates (monotone vs. expressive)
5. Speaking rate — how quickly the person is talking

Ignore background noise (sirens, engines, wind, radio) — focus only on human voices.
Use your assessment of these dimensions to inform your classification below.

Return ONLY a single JSON object — no markdown fences, no text before or after.

{
  "max_volume": <"quiet", "normal", "loud", or "shouting" — peak human voice level>,
  "emotional_tones": [<one or more short lowercase labels, e.g. "calm", "tense", "agitated", "distressed", "angry">]
}

If no speech is present, use: {"max_volume": "quiet", "emotional_tones": []}
}"""


def analyze_prosody(model_id: str, audio_b64: str, audio_fmt: str) -> str:
    """Analyze prosody on pre-encoded audio via Gemini API."""
    audio_bytes = base64.b64decode(audio_b64)
    mime_type = f"audio/{audio_fmt}"
    content = types.Content(
        parts=[
            types.Part(inline_data=types.Blob(data=audio_bytes, mime_type=mime_type)),
            types.Part(text=PROSODY_PROMPT),
        ]
    )
    return call_gemini(model_id, content)

In [ ]:
# Run prosody analysis across all models
prosody_results = []

for i, clip_path in enumerate(prosody_clips):
    print(f"\n{'='*60}")
    print(f"Prosody clip {i}: {prosody_labels[i]}")
    print(f"{'='*60}")

    audio_b64, audio_fmt = prosody_audio_payloads[i]
    results = {}
    for model_name, model_id in AUDIO_MODELS.items():
        print(f"\n--- {model_name} ---")
        results[model_name] = analyze_prosody(model_id, audio_b64, audio_fmt)
        print(results[model_name])

    prosody_results.append(results)

### Prosody Notes

Write your observations here:
- Did it correctly identify shouting vs normal speech?
- Were background sounds detected accurately?
- Is the structured JSON output reliable enough to filter on?

---
## Test 3: OCR (License Plates, Signs, Text)

Compare Gemini Flash vs Claude Sonnet on extracting text from frames.
This is where we hypothesized Sonnet would be more precise.

Requires the **Pre-encode media** cell (`ocr_image_payloads`).

In [ ]:
# Models to compare for OCR
OCR_MODELS = {
    "Gemini 2.5 Flash": "gemini-2.5-flash",
    "Gemini 3 Flash": "gemini-3-flash-preview",
    "GPT-4o-mini": "openai/gpt-4o-mini",
    "Qwen3-VL-32B": "qwen/qwen3-vl-32b-instruct",
}

ocr_results = []

for i, frame_path in enumerate(ocr_frames):
    print(f"\n{'='*60}")
    print(f"OCR frame {i}: {ocr_labels[i]}")
    print(f"{'='*60}")

    img_b64, media_type = ocr_image_payloads[i]
    results = {}
    for model_name, model_id in OCR_MODELS.items():
        print(f"\n--- {model_name} ---")
        results[model_name] = ocr_with_model(model_id, img_b64, media_type)
        print(results[model_name])

    ocr_results.append(results)

In [ ]:
img_b64, media_type = ocr_image_payloads[0]
result = ocr_with_model("gemini-2.5-flash", img_b64, media_type)
print(result)

### OCR Notes

Write your observations:
- Did Sonnet catch text that Flash missed (or vice versa)?
- License plate accuracy difference?
- Did either model hallucinate text that isn't there?
- Is the cost difference justified?

---
## Test 4: Scene Description (Video Clip Quality Check)

Compare 4 models that accept native video input:
- **Gemini 2.5 Flash** — current plan baseline
- **Gemini 3 Flash** — newer Gemini generation
- **Qwen3-VL-32B** — top open-source VLM with native video support
- **NVIDIA Nemotron Nano 2 VL** — 12B model designed specifically for video understanding

Each model receives the full video clip (10-60s with audio) and must return
three structured sections: scene description, transcript, and prosody.

Key questions:
- Which produces the richest, most searchable descriptions?
- Do all models capture temporal events and audio cues?
- Is the cheaper/smaller model good enough?

In [27]:
# Models to compare for scene description (all accept native video input)
# media_resolution only supported on Gemini 3+
VIDEO_MODELS = {
    "Gemini 2.5 Flash": ("gemini-2.5-flash", ["default"]),
    "Gemini 3 Flash": ("gemini-3-flash-preview", ["medium", "high"]),
    #"Qwen3-VL-32B": ("qwen/qwen3-vl-32b-instruct", ["default"]),
    #"Nemotron Nano 2 VL": ("nvidia/nemotron-nano-12b-v2-vl", ["default"]),
}

RESOLUTION_FUNCS = {
    "default": None,  # no media_resolution
    "medium": describe_scene_video_mediumres,
    "high": describe_scene_video_highres,
}

scene_results = []

for i, clip_path in enumerate(scene_clips):
    print(f"\n{'='*60}")
    print(f"Scene {i}: {scene_labels[i]}")
    print(f"{'='*60}")

    video_b64, media_type = scene_video_payloads[i]
    results = {}
    for model_name, (model_id, resolutions) in VIDEO_MODELS.items():
        for res_name in resolutions:
            key = f"{model_name} ({res_name})"
            print(f"\n--- {key} ---")
            if res_name == "default":
                video_bytes = base64.b64decode(video_b64)
                content = types.Content(
                    parts=[
                        types.Part(inline_data=types.Blob(data=video_bytes, mime_type=media_type)),
                        types.Part(text=SCENE_PROMPT),
                    ]
                )
                results[key] = call_gemini(model_id, content)
            else:
                results[key] = RESOLUTION_FUNCS[res_name](model_id, video_b64, media_type)
            print(results[key])

    scene_results.append(results)


Scene 0: lady in purple, man in blue, in the day

--- Gemini 2.5 Flash (default) ---
  [gemini-2.5-flash] 16.8s | 9595 in / 81 out
=== SCENE DESCRIPTION ===
[00:00] A female police officer in a dark blue uniform stands in a parking lot, looking towards a man and a woman who are walking on a concrete sidewalk towards her. [00:06] The officer greets them. [00:08] The officer asks the couple to describe what happened, and the man begins to explain the

--- Gemini 3 Flash (medium) ---
  [gemini-3-flash-preview] 15.6s | 3413 in / 551 out
=== SCENE DESCRIPTION ===
[00:00] A female officer wearing a dark blue Las Cruces police uniform walks through an outdoor parking lot toward an apartment building. She is met by a man and a woman walking toward her on the sidewalk. [00:02] The woman, wearing a purple tank top, black leggings, and sunglasses, introduces her fiancé, Levi. Levi is wearing a light blue short-sleeved button-down shirt and khaki pants. [00:08] The officer asks Levi to describe w

MAX_TOKENS of 2048 is used in this notebook (hence some output being cutoff). Gemini 3 Flash (high) appears to be taking up too many tokens, likely due to the resolution of the video, and hence most of the output is cutoff. Gemini 2.5 Flash seems to be more random with the structured output (giving json for one although not specified) and scene descriptions tend to be more narrative. Hence I will choose to use Gemini 3 Flash (medium) for production.